# Capítulo 15 · Laboratório 15: Gann sem astrologia
*Análise Técnica dos Mercados Financeiros*, 4ª edição, Flávio Lemos, CMT

Dados até **30/09/2026**. Na primeira execução, o notebook baixa os dados do Yahoo Finance na sua máquina e guarda uma cópia local (`dados_cap15.pkl`), que as execuções seguintes reaproveitam. O provedor corrige o histórico (proventos, desdobramentos), por isso seus números podem diferir um pouco dos do livro. O laboratório compara os graus de Gann com os geométricos; mostra que o ângulo 1x1 depende da escala; programa o gráfico de oscilações de 3 dias; testa as retrações de Gann ("passou de 50%" e "vai ao próximo nível"); testa o enquadramento de preço e tempo, as datas das estações e o ciclo lunar; e confere as projeções feitas no podcast em novembro de 2025.

In [ ]:
!pip install -q yfinance

In [ ]:
import logging
import os
import numpy as np
import pandas as pd
import yfinance as yf

logging.getLogger("yfinance").setLevel(logging.CRITICAL)
FIM = "2026-10-01"                      # dados até 30/09/2026
CACHE = "dados_cap15.pkl"
pc = lambda x, k=1: f"{x:.{k}%}".replace(".", ",")
ps = lambda x, k=1: ("+" if x >= 0 else "") + pc(x, k)
nr = lambda x, k=2: f"{x:,.{k}f}".replace(",", "X").replace(".", ",").replace("X", ".")
ACOES = ["PETR4.SA", "VALE3.SA", "ITUB4.SA", "BBDC4.SA", "ABEV3.SA", "WEGE3.SA", "BBAS3.SA", "GGBR4.SA"]


def baixa(t, ini):
    d = yf.download(t, start=ini, end=FIM, auto_adjust=True, progress=False)
    if isinstance(d.columns, pd.MultiIndex):
        d.columns = d.columns.get_level_values(0)
    d = d.dropna(subset=["Close"])
    return d[(d["Close"] > 0) & (d["Low"] > 0)]


def confere(baixados):
    """O Yahoo devolve uma tabela vazia quando falha: para aqui, com uma mensagem clara, sem gravar a cópia local."""
    for nome, x in baixados.items():
        if len(x) == 0:
            raise RuntimeError(f"Não consegui baixar {nome} do Yahoo Finance. O provedor pode estar fora do ar ou"
                               " limitando pedidos. Espere alguns minutos e rode a célula de novo.")


D = pd.read_pickle(CACHE) if os.path.exists(CACHE) else {}
for nome, (t, ini) in {"Ibovespa": ("^BVSP", "1995-01-01"), "S&P 500": ("^GSPC", "1950-01-01"),
                       "Ouro": ("GC=F", "2020-01-01"), "Intel": ("INTC", "2020-01-01"),
                       **{a: (a, "2000-01-01") for a in ACOES}}.items():
    if nome not in D:
        D[nome] = baixa(t, ini)
confere({k: D[k] for k in ("Ibovespa", "S&P 500", "Ouro", "Intel")})
pd.to_pickle(D, CACHE)
rng = np.random.default_rng(42)


def zigzag(c, limite):
    """Pivôs de fechamento: um topo (fundo) se confirma quando o preço cai (sobe) 'limite' a partir dele.
    Devolve lista de (data, preço, tipo), com tipo +1 para topo e -1 para fundo."""
    pivos, modo = [], 0
    hi = lo = c.iloc[0]
    hi_i = lo_i = c.index[0]
    for d, p in c.items():
        if modo >= 0 and p > hi:
            hi, hi_i = p, d
        if modo <= 0 and p < lo:
            lo, lo_i = p, d
        if modo >= 0 and p <= hi * (1 - limite):
            pivos.append((hi_i, hi, 1))
            modo, lo, lo_i = -1, p, d
        elif modo <= 0 and p >= lo * (1 + limite):
            pivos.append((lo_i, lo, -1))
            modo, hi, hi_i = 1, p, d
    return pivos

## Parte 1: dados

In [ ]:
IB, SPX = D["Ibovespa"], D["S&P 500"]
print(f"Parte 1. Ibovespa desde {IB.index[0]:%Y}, S&P 500 desde {SPX.index[0]:%Y}, {len(ACOES)} ações da B3 desde 2000,"
      f" ouro e Intel desde 2020; último pregão {IB.index[-1]:%d/%m/%Y}.")

## Parte 2: graus de nome x graus geométricos

In [ ]:
ANGULOS = [("1 x 8", 8), ("1 x 4", 4), ("1 x 3", 3), ("1 x 2", 2), ("1 x 1", 1), ("2 x 1", 1 / 2), ("3 x 1", 1 / 3),
           ("4 x 1", 1 / 4), ("8 x 1", 1 / 8)]
NOME = {"1 x 8": 82.5, "1 x 4": 75, "1 x 3": 71.25, "1 x 2": 63.75, "1 x 1": 45, "2 x 1": 26.25, "3 x 1": 18.75,
        "4 x 1": 15, "8 x 1": 7.5}
print("\nParte 2. Tempo x preço: grau usado por Gann e grau geométrico (com um quadradinho de preço igual a um de tempo)")
TAB_ANG = []
for rot, r in ANGULOS:
    geo = np.degrees(np.arctan(r))
    TAB_ANG.append((rot, r, NOME[rot], geo))
    print(f"   {rot}: {nr(r, 3):>6} unidade(s) de preço por unidade de tempo | Gann {nr(NOME[rot], 2):>5}° | geométrico"
          f" {nr(geo, 2):>5}°")

## Parte 3: o 1x1 depende da escala

In [ ]:
SEM = IB["Close"].resample("W-FRI").last().dropna()
FUNDO = SEM.loc["2020"].idxmin()
ALTA_S, BAIXA_S = IB["High"].resample("W-FRI").max(), IB["Low"].resample("W-FRI").min()
FAIXA = (ALTA_S - BAIXA_S).loc[:FUNDO].iloc[-53:-1].mean()      # amplitude semanal média do ano anterior ao fundo
DEPOIS = SEM.loc[FUNDO:]
ESCALAS = [("100 pontos por semana", 100), ("500 pontos por semana", 500),
           (f"1 amplitude semanal média ({nr(FAIXA, 0)} pontos)", FAIXA)]
print(f"\nParte 3. Ângulo 1x1 a partir do fundo semanal do Ibovespa em {FUNDO:%d/%m/%Y} ({nr(SEM[FUNDO], 0)} pontos)")
TAB_ESC = []
for nome, u in ESCALAS:
    linha = SEM[FUNDO] + u * np.arange(len(DEPOIS))
    abaixo = DEPOIS.values < linha
    quebra = DEPOIS.index[abaixo.argmax()] if abaixo.any() else None
    TAB_ESC.append((nome, u, linha[-1], DEPOIS.iloc[-1] > linha[-1], quebra))
    print(f"   {nome:<42} linha hoje em {nr(linha[-1], 0):>9} | preço {'acima' if DEPOIS.iloc[-1] > linha[-1] else 'abaixo'}"
          f" | primeiro fechamento abaixo: {'nunca' if quebra is None else f'{quebra:%d/%m/%Y}'}")

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 3.5))
plt.plot(SEM.loc[FUNDO:].index, SEM.loc[FUNDO:].values, color="black")
for nome, u, *_ in TAB_ESC:
    plt.plot(SEM.loc[FUNDO:].index, SEM[FUNDO] + u * np.arange(len(SEM.loc[FUNDO:])), "--", label=nome)
plt.ylim(50000, 250000)
plt.legend(fontsize=7)
plt.title("O mesmo 1x1 em três escalas")
plt.show()

## Parte 4: swing chart de 3 dias

In [ ]:
def swing(d, n=3):
    """Gráfico de oscilações de Gann (máximas e mínimas, não fechamentos). A linha vira para cima depois de n dias
    seguidos de máximas e mínimas mais altas e para baixo depois de n dias seguidos de mínimas mais baixas.
    Devolve a direção (+1/-1) em cada dia e a lista de pivôs (data, preço, tipo)."""
    h, l = d["High"].to_numpy(), d["Low"].to_numpy()
    sobe = (h > np.roll(h, 1)) & (l > np.roll(l, 1))
    desce = l < np.roll(l, 1)
    sobe[0] = desce[0] = False
    dirc, pivos = np.zeros(len(d), int), []
    modo, ext, ext_i, cs, cd = 0, h[0], 0, 0, 0
    for i in range(1, len(d)):
        cs = cs + 1 if sobe[i] else 0
        cd = cd + 1 if desce[i] else 0
        if modo >= 0 and h[i] > ext:
            ext, ext_i = h[i], i
        if modo <= 0 and l[i] < ext:
            ext, ext_i = l[i], i
        if modo >= 0 and cd >= n:
            if modo == 1:
                pivos.append((d.index[ext_i], ext, 1))
            modo, ext, ext_i = -1, l[i - n + 1:i + 1].min(), i - n + 1 + l[i - n + 1:i + 1].argmin()
        elif modo <= 0 and cs >= n:
            if modo == -1:
                pivos.append((d.index[ext_i], ext, -1))
            modo, ext, ext_i = 1, h[i - n + 1:i + 1].max(), i - n + 1 + h[i - n + 1:i + 1].argmax()
        dirc[i] = modo
    return pd.Series(dirc, d.index), pivos


IB00 = IB.loc["2000":]
DIR3, PIV3 = swing(IB00, 3)
DUR = np.diff([IB00.index.get_loc(p[0]) for p in PIV3])
AMP = np.abs(np.diff([p[1] for p in PIV3])) / np.array([p[1] for p in PIV3[:-1]])
R20 = IB00["Close"].shift(-20) / IB00["Close"] - 1
CIMA, BAIXO = R20[DIR3 == 1].dropna(), R20[DIR3 == -1].dropna()
print(f"\nParte 4. Swing chart de 3 dias no Ibovespa desde 2000: {len(PIV3)} pivôs, oscilação mediana de {np.median(DUR):.0f}"
      f" pregões e {pc(np.median(AMP))}")
SEP = R20.iloc[::20].dropna()                                  # um dia a cada 20, para as janelas não se sobreporem
A4, B4 = SEP[DIR3.loc[SEP.index] == 1], SEP[DIR3.loc[SEP.index] == -1]
T4 = (A4.mean() - B4.mean()) / np.sqrt(A4.var() / len(A4) + B4.var() / len(B4))
print(f"   Retorno dos 20 pregões seguintes com a linha de 3 dias para cima: média {ps(CIMA.mean(), 2)} |"
      f" para baixo: {ps(BAIXO.mean(), 2)} | todos os dias: {ps(R20.mean(), 2)} | estatística t (janelas sem"
      f" sobreposição): {nr(T4)}")

## Parte 5: retrações de Gann

In [ ]:
# Para cada perna A-B do zigzag de 10%, segue o preço depois de B até o primeiro de dois desfechos: nova máxima além de B
# ou devolução de 100% (volta a A). Guarda a maior fração devolvida antes do desfecho e a fração que a própria confirmação
# do pivô B já obriga (10% contra a perna), para não contar um nível que o método cruzou por construção.
LIMITE = 0.10
SERIES = {"Ibovespa": IB["Close"], "S&P 500": SPX["Close"], **{a: D[a]["Close"] for a in ACOES}}
PERNAS = []
for t, c in SERIES.items():
    pv = zigzag(c, LIMITE)
    for (d0, p0, _), (d1, p1, tipo) in zip(pv, pv[1:]):
        s = tipo
        caminho = c.loc[d1:].iloc[1:] * s
        nova, volta = caminho > p1 * s, caminho <= p0 * s
        if not (nova.any() or volta.any()):
            continue
        fim = min(caminho.index[nova.argmax()] if nova.any() else caminho.index[-1],
                  caminho.index[volta.argmax()] if volta.any() else caminho.index[-1])
        trecho = c.loc[d1:fim] * s
        devolveu = min((trecho.min() - p1 * s) / ((p0 - p1) * s), 1.0)
        obrigado = LIMITE * p1 / abs(p1 - p0)
        cem = bool(volta.any() and caminho.index[volta.argmax()] == fim)
        PERNAS.append((t, devolveu, obrigado, cem))
PERNAS = pd.DataFrame(PERNAS, columns=["ativo", "devolveu", "obrigado", "cem"])
NIVEIS = [3 / 8, 1 / 2, 5 / 8, 3 / 4, 7 / 8]
BASE5 = PERNAS[PERNAS["obrigado"] < NIVEIS[0]]                 # pernas em que a confirmação não obriga a cruzar 37,5%
TAB_50, TAB_PROX = [], []
print(f"\nParte 5. {len(BASE5)} pernas grandes (Ibovespa, S&P 500 e {len(ACOES)} ações da B3, zigzag de 10%), em que"
      f" a confirmação do pivô não obriga a correção a passar de 37,5%")
print(f"   Todas: devolveram 100% antes de nova máxima em {pc(BASE5['cem'].mean(), 0)}")
for k, nv in enumerate(NIVEIS):
    x = BASE5[BASE5["devolveu"] >= nv]
    prox = NIVEIS[k + 1] if k + 1 < len(NIVEIS) else 1.0
    TAB_50.append((nv, len(x), x["cem"].mean()))
    TAB_PROX.append((nv, prox, len(x), (x["devolveu"] >= prox).mean()))
    print(f"   Passaram de {nr(nv * 100, 1)}%: {len(x):>3} | devolveram 100%: {pc(x['cem'].mean(), 0)} | chegaram ao"
          f" nível seguinte ({nr(prox * 100, 1)}%): {pc((x['devolveu'] >= prox).mean(), 0)}")

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 3.5))
plt.bar([f"{nv:.1%}" for nv, *_ in TAB_50], [x[2] * 100 for x in TAB_50], color="gray")
plt.ylabel("devolveram 100% (%)")
plt.title("Correções que passaram de cada nível de Gann")
plt.show()

## Parte 6: enquadramento de preço e tempo

In [ ]:
# Escala: 1 unidade de preço = a amplitude semanal média das 52 semanas antes de B. O "quadrado fecha" quando o número
# de semanas depois de B iguala o tamanho da perna A-B medido nessa unidade. Acerto: um pivô do zigzag a até 3 semanas
# dessa data. Comparação: as mesmas distâncias sorteadas entre as pernas (o acaso com a mesma distribuição de prazos).
SEMH = IB.resample("W-FRI").agg({"High": "max", "Low": "min", "Close": "last"}).dropna()
AMPS = (SEMH["High"] - SEMH["Low"]).rolling(52).mean()
PV_S = zigzag(SEMH["Close"], 0.15)
DATAS_PV = np.array([SEMH.index.get_loc(p[0]) for p in PV_S])
FECHA, BASES = [], []
for (d0, p0, _), (d1, p1, _) in zip(PV_S, PV_S[1:]):
    i1 = SEMH.index.get_loc(d1)
    if np.isnan(AMPS.iloc[i1]):
        continue
    semanas = int(round(abs(p1 - p0) / AMPS.iloc[i1]))
    if i1 + semanas + 3 < len(SEMH):
        FECHA.append(semanas)
        BASES.append(i1)
FECHA, BASES = np.array(FECHA), np.array(BASES)


def acertos(bases, prazos, j=3):
    return np.mean([np.any(np.abs(DATAS_PV - (b + s)) <= j) for b, s in zip(bases, prazos)])


ACERTO_Q = acertos(BASES, FECHA)
ACASO_Q = np.mean([acertos(BASES, rng.permutation(FECHA)) for _ in range(2000)])
print(f"\nParte 6. Enquadramento no Ibovespa semanal ({len(FECHA)} pernas do zigzag de 15%, prazo mediano de"
      f" {np.median(FECHA):.0f} semanas): pivô a até 3 semanas da data do quadrado em {pc(ACERTO_Q, 0)} | com os prazos"
      f" sorteados: {pc(ACASO_Q, 0)}")

## Parte 7: datas sazonais de Gann

In [ ]:
ESTACOES = [(3, 21), (6, 21), (9, 23), (12, 21)]
PV_SP = zigzag(SPX["Close"], 0.05)


def perto_data(d, janela=5):
    return any(abs((d - pd.Timestamp(d.year + k, m, dia)).days) <= janela for m, dia in ESTACOES for k in (-1, 0, 1))


ACHOU = np.mean([perto_data(d) for d, _, _ in PV_SP])
ESPERADO = 4 * 11 / 365.25
print(f"\nParte 7. {len(PV_SP)} topos e fundos do S&P 500 desde 1950 (zigzag de 5%): a até 5 dias de um equinócio ou"
      f" solstício em {pc(ACHOU, 0)} | esperado pelo acaso: {pc(ESPERADO, 0)}")

## Parte 8: ciclo lunar

In [ ]:
LUA_NOVA = pd.Timestamp("2000-01-06 18:14")          # lua nova de referência (UTC)
MES_SINODICO = 29.530588853                          # duração média do ciclo lunar (dias); erro de algumas horas
TAB_LUA = []
print("\nParte 8. Retorno diário perto da lua nova (7 dias antes e depois) e perto da lua cheia")
for nome, d in [("Ibovespa", IB), ("S&P 500", SPX)]:
    r = np.log(d["Close"]).diff().dropna()
    fase = ((r.index - LUA_NOVA).total_seconds() / 86400 / MES_SINODICO) % 1
    nova = (fase < 0.25) | (fase >= 0.75)
    a, b = r[nova], r[~nova]
    t = (a.mean() - b.mean()) / np.sqrt(a.var() / len(a) + b.var() / len(b))
    TAB_LUA.append((nome, a.mean() * 252, b.mean() * 252, t))
    print(f"   {nome:<9} lua nova: {ps(a.mean() * 252)} ao ano | lua cheia: {ps(b.mean() * 252)} ao ano | estatística t:"
          f" {nr(t)}")

## Parte 9: o teste do tempo

In [ ]:
S = SPX["Close"]
GRAV = pd.Timestamp("2025-11-24")                    # data provável da gravação (a conferir no canal)
BASE = S.loc[:GRAV].iloc[-1]
DEZ = S.loc[GRAV:"2025-12-19"].max()
A26 = S.loc["2026"]
PRIM = lambda nivel: (S.loc[GRAV:] >= nivel).idxmax() if (S.loc[GRAV:] >= nivel).any() else None
OURO, INTEL = D["Ouro"]["Close"], D["Intel"]["Close"]
print(f"\nParte 9. O teste do tempo (projeções de Haroldo MacDowell, gravação provável em {GRAV:%d/%m/%Y})")
print(f"   S&P 500 em {S.loc[:GRAV].index[-1]:%d/%m/%Y}: {nr(BASE, 0)} | máxima de fechamento até o vencimento de"
      f" 19/12/2025: {nr(DEZ, 0)} (o mercado de opções apontava perto de 7.000)")
for nivel in (7600, 7931):
    p = PRIM(nivel)
    print(f"   Primeiro fechamento a partir de {nr(nivel, 0)}: {'ainda não' if p is None else f'{p:%d/%m/%Y}'}")
print(f"   Máxima de fechamento em 2026 (até {S.index[-1]:%d/%m/%Y}): {nr(A26.max(), 0)} em {A26.idxmax():%d/%m/%Y} |"
      f" último: {nr(S.iloc[-1], 0)}")
print(f"   Ouro (contrato futuro) em {OURO.index[-1]:%d/%m/%Y}: US$ {nr(OURO.iloc[-1], 0)} (alvo de cerca de 6.400 até"
      f" 2028; em {GRAV:%d/%m/%Y}: US$ {nr(OURO.loc[:GRAV].iloc[-1], 0)})")
print(f"   Intel em {INTEL.index[-1]:%d/%m/%Y}: US$ {nr(INTEL.iloc[-1])} (alvo de cerca de 96; em {GRAV:%d/%m/%Y}:"
      f" US$ {nr(INTEL.loc[:GRAV].iloc[-1])}; primeiro fechamento a partir de 96:"
      f" {(INTEL.loc[GRAV:] >= 96).idxmax():%d/%m/%Y})")

## Peça à IA

O Colab tem um assistente de IA. Abra o assistente, cole um destes pedidos e confira a resposta rodando a célula indicada.

1. **Explicar:** "Explique, linha por linha, a função `swing` da Parte 4. Mostre onde estão as condições de máxima e mínima mais altas (`sobe`) e de mínima mais baixa (`desce`), para que serve o `np.roll`, por que a primeira posição é zerada e por que a regra usa máximas e mínimas, e não fechamentos." Confira com a Parte 4, que imprime 419 pivôs no Ibovespa desde 2000, com oscilação mediana de 12 pregões e 8,9%.
2. **Adaptar:** "Acrescente [ATIVO] ao dicionário da célula de abertura, com o ticker do Yahoo Finance e a data inicial. Numa célula nova, depois da Parte 4, rode `swing(D["[ATIVO]"], 3)` e imprima o número de pivôs e a oscilação mediana, como a Parte 4 faz. Não mude o gráfico de 3 dias." Antes de olhar o ativo novo, confira se a Parte 4 continua com 419 pivôs no Ibovespa, como no livro.

Mais pedidos deste capítulo (entender e aplicar ao seu gráfico) estão na página do capítulo, no hub do livro.

## Para ir além
1. Na Parte 3, use a escala logarítmica: um 1x1 de 1% por semana. Quando o Ibovespa perdeu a linha?
2. Na Parte 4, troque o gráfico de 3 dias pelo de 2 dias. O número de pivôs muda quanto? E a diferença de retorno entre linha para cima e para baixo?
3. Na Parte 6, use metade da amplitude semanal como unidade. O enquadramento passa a acertar mais que o acaso?

**Governança:** os resultados deste notebook valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo. 

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros. 

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido. 

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.